# Sector-Rotation Training (Relative Targets)

Trains one LSTM per **(sector × horizon)** combination.

**Key change from previous run:** targets are now *relative* — a sector is labelled 1
if its forward return beats the median of all 8 sectors on the same day, 0 otherwise.
This gives ~50/50 class balance by construction, removing the positive market-drift
bias that inflated accuracy (and faked AUC) in the absolute-target run.

**Architecture:** same `SentimentLSTM(input=32, hidden=32, layers=2)` as per-stock.  
**Scheduler:** `ReduceLROnPlateau` — decays only when val_loss stops improving.

In [1]:
from __future__ import annotations

import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import src
from src.log import setup_logging

setup_logging()
logger = logging.getLogger("train_sector")

## Config

In [2]:
from src.training import ComputeConfig, TrainingConfig

CUTOFF      = "2023-06-01"
VAL_FRAC    = 0.1
PRICE_YEARS = list(range(2018, 2025))
HORIZONS    = [5, 10, 21, 42]
WINDOW      = 20
SEED        = 42

config = TrainingConfig(
    window=WINDOW,
    batch_size=16,
    n_epochs=150,
    lr=1e-3,
    weight_decay=1e-4,
    patience=20,
    scheduler="plateau",
    scheduler_patience=10,
    grad_clip=1.0,
    seed=SEED,
)

compute_config = ComputeConfig(num_workers=0)
compute_config.setup()

print(f"Device    : {compute_config.device}")
print(f"Horizons  : {HORIZONS}")
print(f"Scheduler : {config.scheduler} (patience={config.scheduler_patience})")
print(f"Epochs    : {config.n_epochs}  early-stop patience={config.patience}")

Device    : cpu
Horizons  : [5, 10, 21, 42]
Scheduler : plateau (patience=10)
Epochs    : 150  early-stop patience=20


## Load price and sentiment data

In [3]:
from src.features.sectors import SECTORS
from src.repositories.prices import PriceRepository
from src.repositories.sentiment import SentimentRepository

PRICE_DIR = Path("../data/historical-prices/prices/data/historical-prices")
SENT_DIR  = Path("../data/sentiment/data/sentiment")

price_repo = PriceRepository(data_dir=PRICE_DIR)
sent_repo  = SentimentRepository(data_dir=SENT_DIR)

all_tickers = sorted({t for tickers in SECTORS.values() for t in tickers})

price_data:     dict[str, pd.DataFrame] = {}
sentiment_data: dict[str, pd.DataFrame] = {}

for ticker in all_tickers:
    try:
        price_data[ticker]     = price_repo.load_years(ticker, PRICE_YEARS)
        sentiment_data[ticker] = sent_repo.load(ticker)
    except FileNotFoundError:
        print(f"  Missing: {ticker}")

print(f"Loaded {len(price_data)} tickers")

Loaded 48 tickers


## Pre-build sector price indices

Build each sector's equal-weight price index once and reuse it across all horizons.
This also lets us compute cross-sector relative labels before the training loop.

In [4]:
from src.features.sectors import build_sector_price_index

price_indices: dict[str, pd.DataFrame] = {}
sector_tickers: dict[str, list[str]]   = {}

for sector_name, tickers in SECTORS.items():
    available = [t for t in tickers if t in price_data]
    if len(available) < 2:
        print(f"Skipping {sector_name}: only {len(available)} tickers")
        continue
    price_indices[sector_name]  = build_sector_price_index({t: price_data[t] for t in available})
    sector_tickers[sector_name] = available
    print(f"{sector_name:<20} {len(available)} tickers  {len(price_indices[sector_name])} trading days")

Technology           10 tickers  1756 trading days
Healthcare           6 tickers  1756 trading days
Financials           6 tickers  1756 trading days
Energy               4 tickers  1756 trading days
ConsumerDisc         7 tickers  1756 trading days
ConsumerStaples      4 tickers  1756 trading days
Industrials          5 tickers  1756 trading days
UtilTelecom          6 tickers  1546 trading days


## Sanity check: class balance under relative targets

Each horizon should give ~50/50 positive rate across all sectors.
If any sector shows a strongly skewed rate, something is wrong with the label computation.

In [5]:
from src.features.sectors import compute_cross_sector_labels

print("Positive-rate sanity check (should be ~0.50 everywhere)\n")
print(f"{'Sector':<20}  " + "  ".join(f"T+{h:2d}" for h in HORIZONS))
print("-" * 60)

for sector_name in price_indices:
    rates = []
    for horizon in HORIZONS:
        labels = compute_cross_sector_labels(price_indices, horizon)
        lbl    = labels[sector_name]
        valid  = lbl[lbl >= 0]
        rates.append(f"{valid.mean():.3f}")
    print(f"{sector_name:<20}  " + "  ".join(rates))

Positive-rate sanity check (should be ~0.50 everywhere)

Sector                T+ 5  T+10  T+21  T+42
------------------------------------------------------------
Technology            0.544  0.570  0.568  0.582
Healthcare            0.493  0.517  0.531  0.517
Financials            0.509  0.531  0.549  0.604
Energy                0.446  0.429  0.416  0.398
ConsumerDisc          0.533  0.531  0.511  0.509
ConsumerStaples       0.541  0.547  0.583  0.598
Industrials           0.467  0.447  0.410  0.388
UtilTelecom           0.466  0.429  0.433  0.406


## Train: horizon × sector grid

Outer loop is **horizon** so cross-sector labels are computed once per horizon
and reused across all 8 sectors.

In [7]:
from src.features.sectors import SectorDataset, build_sector_loaders
from src.model.lstm import SentimentLSTM
from src.model.trainer import Trainer
from src.repositories.models import ModelRepository

model_repo = ModelRepository()
records: list[dict] = []

for horizon in HORIZONS:
    print(f"\n{'#'*60}")
    print(f"# HORIZON = T+{horizon}")
    print(f"{'#'*60}")

    # Compute relative labels once for all sectors at this horizon
    cross_labels = compute_cross_sector_labels(price_indices, horizon)

    for sector_name in price_indices:
        print(f"\n{'='*60}")
        print(f"{sector_name}  |  horizon=T+{horizon}")
        print(f"{'='*60}")

        available    = sector_tickers[sector_name]
        sec_prices   = {t: price_data[t]     for t in available}
        sec_sent     = {t: sentiment_data[t] for t in available}

        try:
            ds = SectorDataset(
                name=sector_name,
                price_dfs=sec_prices,
                sentiment_dfs=sec_sent,
                window=WINDOW,
                horizon=horizon,
                target_labels=cross_labels[sector_name],
            )
        except RuntimeError as exc:
            print(f"  Dataset error: {exc}")
            continue

        train_loader, val_loader, test_loader = build_sector_loaders(
            ds, cutoff=CUTOFF, val_frac=VAL_FRAC, batch_size=config.batch_size,
        )

        n_train = len(train_loader.dataset)
        n_val   = len(val_loader.dataset)
        n_test  = len(test_loader.dataset)
        pos_rate = ds.y.mean()
        print(f"  Windows — train: {n_train}, val: {n_val}, test: {n_test}  pos_rate={pos_rate:.3f}")

        if n_train == 0 or n_test == 0:
            print("  Skipping: empty split")
            continue

        model = SentimentLSTM(
            n_factors=16, sentiment_dim=768, hidden_size=32, num_layers=2, dropout=0.2,
        )
        trainer      = Trainer(model, config, compute_config)
        train_result = trainer.fit(train_loader, val_loader)

        print(
            f"  Best epoch: {train_result.best_epoch} | "
            f"val_loss: {train_result.best_val_loss:.4f} | "
            f"val_auc: {train_result.best_val_auc:.4f}"
        )

        eval_result = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=SEED)
        print(
            f"  Test AUC: {eval_result.auc_mean:.3f} "
            f"[{eval_result.auc_ci_low:.3f}, {eval_result.auc_ci_high:.3f}]"
        )
        print(
            f"  Test Acc: {eval_result.accuracy_mean:.3f} "
            f"[{eval_result.accuracy_ci_low:.3f}, {eval_result.accuracy_ci_high:.3f}]"
        )

        model_repo.save(
            f"sector_rel_{sector_name}_T{horizon}",
            model,
            {
                "sector": sector_name, "tickers": available,
                "horizon": horizon, "window": WINDOW, "mode": "relative",
                "n_train": n_train, "n_val": n_val, "n_test": n_test,
                "best_epoch": train_result.best_epoch,
                "best_val_loss": train_result.best_val_loss,
                "best_val_auc": train_result.best_val_auc,
                "test_auc": eval_result.auc_mean,
                "test_accuracy": eval_result.accuracy_mean,
                "history": train_result.history,
            },
        )

        records.append({
            "sector":      sector_name,
            "horizon":     horizon,
            "n_train":     n_train,
            "n_test":      n_test,
            "best_epoch":  train_result.best_epoch,
            "val_auc":     train_result.best_val_auc,
            "test_auc":    eval_result.auc_mean,
            "auc_ci_low":  eval_result.auc_ci_low,
            "auc_ci_high": eval_result.auc_ci_high,
            "test_acc":    eval_result.accuracy_mean,
        })

print(f"\n\nDone. {len(records)} models trained.")

15:49:43 INFO     src.features.sectors  Technology  horizon=T+5  mode=relative | 1522 windows | pos_rate=0.55 | 96% days with news
15:49:43 INFO     src.features.sectors  Technology  horizon=T+5 | train=1017  val=112  test=393



############################################################
# HORIZON = T+5
############################################################

Technology  |  horizon=T+5
  Windows — train: 1017, val: 112, test: 393  pos_rate=0.547


15:49:44 INFO     src.model.trainer  Epoch   1 | train_loss=0.8413 | val_loss=0.6635 | val_auc=0.5595 | val_acc=0.6250
15:49:44 INFO     src.model.trainer  Epoch   2 | train_loss=0.7347 | val_loss=0.6898 | val_auc=0.3245 | val_acc=0.6250
15:49:44 INFO     src.model.trainer  Epoch   3 | train_loss=0.7203 | val_loss=0.7832 | val_auc=0.4112 | val_acc=0.3750
15:49:44 INFO     src.model.trainer  Epoch   4 | train_loss=0.7161 | val_loss=0.6889 | val_auc=0.3381 | val_acc=0.6339
15:49:45 INFO     src.model.trainer  Epoch   5 | train_loss=0.6894 | val_loss=0.7221 | val_auc=0.3340 | val_acc=0.3571
15:49:46 INFO     src.model.trainer  Epoch   6 | train_loss=0.6812 | val_loss=0.7505 | val_auc=0.3371 | val_acc=0.3750
15:49:46 INFO     src.model.trainer  Epoch   7 | train_loss=0.6788 | val_loss=0.7284 | val_auc=0.3296 | val_acc=0.4286
15:49:47 INFO     src.model.trainer  Epoch   8 | train_loss=0.6666 | val_loss=0.8280 | val_auc=0.3946 | val_acc=0.3661
15:49:48 INFO     src.model.trainer  Epoch   9 |

  Best epoch: 1 | val_loss: 0.6635 | val_auc: 0.5595


15:49:58 INFO     src.features.sectors  Healthcare  horizon=T+5  mode=relative | 1522 windows | pos_rate=0.49 | 95% days with news
15:49:58 INFO     src.features.sectors  Healthcare  horizon=T+5 | train=1017  val=112  test=393


...

  Best epoch: 14 | val_loss: 0.5991 | val_auc: 0.7284


15:59:11 INFO     src.features.sectors  UtilTelecom  horizon=T+42  mode=relative | 1397 windows | pos_rate=0.39 | 82% days with news
15:59:11 INFO     src.features.sectors  UtilTelecom  horizon=T+42 | train=937  val=104  test=356


  Test AUC: 0.265 [0.216, 0.318]
  Test Acc: 0.534 [0.480, 0.587]

UtilTelecom  |  horizon=T+42
  Windows — train: 937, val: 104, test: 356  pos_rate=0.393


15:59:12 INFO     src.model.trainer  Epoch   1 | train_loss=0.7567 | val_loss=0.7969 | val_auc=0.4046 | val_acc=0.3173
15:59:12 INFO     src.model.trainer  Epoch   2 | train_loss=0.7214 | val_loss=0.5870 | val_auc=0.6175 | val_acc=0.7212
15:59:13 INFO     src.model.trainer  Epoch   3 | train_loss=0.6591 | val_loss=0.6862 | val_auc=0.4818 | val_acc=0.6154
15:59:14 INFO     src.model.trainer  Epoch   4 | train_loss=0.5654 | val_loss=0.7281 | val_auc=0.6037 | val_acc=0.5769
15:59:14 INFO     src.model.trainer  Epoch   5 | train_loss=0.4973 | val_loss=0.7997 | val_auc=0.6777 | val_acc=0.7212
15:59:15 INFO     src.model.trainer  Epoch   6 | train_loss=0.4266 | val_loss=1.0537 | val_auc=0.6124 | val_acc=0.4519
15:59:15 INFO     src.model.trainer  Epoch   7 | train_loss=0.3869 | val_loss=0.7290 | val_auc=0.6244 | val_acc=0.7500
15:59:16 INFO     src.model.trainer  Epoch   8 | train_loss=0.3517 | val_loss=0.8844 | val_auc=0.6671 | val_acc=0.6827
15:59:17 INFO     src.model.trainer  Epoch   9 |

  Best epoch: 2 | val_loss: 0.5870 | val_auc: 0.6175
  Test AUC: 0.583 [0.523, 0.642]
  Test Acc: 0.595 [0.542, 0.649]


Done. 32 models trained.


## Results

In [8]:
df = pd.DataFrame(records)

auc_pivot = df.pivot(index="sector", columns="horizon", values="test_auc")
auc_pivot.columns = [f"T+{h}" for h in auc_pivot.columns]
auc_pivot["best"] = auc_pivot.max(axis=1)
auc_pivot = auc_pivot.sort_values("best", ascending=False)

print("=== Test AUC Matrix (relative targets) ===")
print(auc_pivot.to_string(float_format="%.3f"))
print(f"\nColumn means:")
print(auc_pivot.drop(columns="best").mean().to_string(float_format="%.3f"))
print(f"\nModels with AUC > 0.55: {(df['test_auc'] > 0.55).sum()} / {len(df)}")
print(f"Models with AUC > 0.50: {(df['test_auc'] > 0.50).sum()} / {len(df)}")

=== Test AUC Matrix (relative targets) ===
                  T+5  T+10  T+21  T+42  best
sector                                       
Healthcare      0.531 0.554 0.464 0.617 0.617
Energy          0.442 0.477 0.484 0.602 0.602
UtilTelecom     0.576 0.536 0.455 0.583 0.583
Industrials     0.502 0.557 0.433 0.265 0.557
Financials      0.524 0.474 0.417 0.463 0.524
Technology      0.348 0.426 0.505 0.390 0.505
ConsumerDisc    0.470 0.445 0.490 0.503 0.503
ConsumerStaples 0.373 0.391 0.378 0.346 0.391

Column means:
T+5    0.471
T+10   0.483
T+21   0.453
T+42   0.471

Models with AUC > 0.55: 6 / 32
Models with AUC > 0.50: 12 / 32


In [9]:
# Full detail — epoch column tells us whether models actually trained
detail = df.sort_values("test_auc", ascending=False).copy()
detail["ci"] = detail.apply(
    lambda r: f"[{r['auc_ci_low']:.3f}, {r['auc_ci_high']:.3f}]", axis=1
)
print(detail[["sector", "horizon", "n_train", "n_test",
              "best_epoch", "val_auc", "test_auc", "ci", "test_acc"]]
      .to_string(index=False, float_format="%.3f"))

         sector  horizon  n_train  n_test  best_epoch  val_auc  test_auc             ci  test_acc
     Healthcare       42     1017     356           1    0.390     0.617 [0.560, 0.673]     0.570
         Energy       42     1017     356           9    0.584     0.602 [0.545, 0.662]     0.664
    UtilTelecom       42      937     356           2    0.617     0.583 [0.523, 0.642]     0.595
    UtilTelecom        5      937     393           1    0.558     0.576 [0.517, 0.631]     0.595
    Industrials       10     1017     388           1    0.604     0.557 [0.500, 0.613]     0.611
     Healthcare       10     1017     388           7    0.400     0.554 [0.498, 0.611]     0.533
    UtilTelecom       10      937     388           5    0.705     0.536 [0.472, 0.597]     0.612
     Healthcare        5     1017     393           3    0.485     0.531 [0.469, 0.591]     0.525
     Financials        5     1017     393           8    0.482     0.524 [0.464, 0.582]     0.438
     Technology     